# Equity Factor Analysis and Return Prediction

A reproducible, leakage-aware study of whether trailing technical factors contain cross-sectional information about next-five-day U.S. equity returns. The notebook uses the functions in `analysis.py`; the command-line script generates the permanent result files.

## Research design

- Select 120 liquid names using median dollar volume measured before 2017.
- Construct only trailing return, volatility, volume, and range features.
- Train before 2017, validate in the first half of 2017, and test from July 2017 onward.
- Choose the Ridge penalty using validation rank IC; inspect the test set once.

In [ ]:
from analysis import download_data, build_panel, FEATURES
import pandas as pd

download_data()
panel, data_summary = build_panel()
pd.Series(data_summary, name='value')

In [ ]:
train = panel[panel['date'] < '2017-01-01']
validation = panel[(panel['date'] >= '2017-01-01') & (panel['date'] < '2017-07-01')]
test = panel[panel['date'] >= '2017-07-01']

pd.DataFrame({
    'period': ['train', 'validation', 'test'],
    'rows': [len(train), len(validation), len(test)],
    'start': [x['date'].min().date() for x in [train, validation, test]],
    'end': [x['date'].max().date() for x in [train, validation, test]],
})

## Reproduce final outputs

The following cell re-runs model selection and writes coefficients, metrics, predictions, and the JSON summary to `results/`.

In [ ]:
from analysis import main
main()

In [ ]:
metrics = pd.read_csv('results/metrics.csv')
metrics

## Interpretation

The simple factor set does not generalize: test R-squared and mean daily rank IC are slightly negative, while positive quintile-spread days are close to 50%. This is a useful negative result. It shows that apparent technical relationships should not be described as alpha without stable performance on untouched chronological data. See `METHODOLOGY.md` for limitations, including survivorship bias, corporate-action handling, the short test window, and omitted trading costs.